In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import sys
import json

import numpy as np
import pandas as pd

BASE_DIR = Path("/content/drive/MyDrive/Legal_Sytem")
DATA_DIR = BASE_DIR / "01_Dataset/02_processed_data"
RETRIEVER_DIR = BASE_DIR / "02_Retriever"

BM25_OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/BM25/output"
BIENCODER_OUTPUT_DIR = RETRIEVER_DIR / "02_Baseline/BiEncoder/output"

TRAIN_DATA_DIR = RETRIEVER_DIR / "03_Finetuning/train_data"
TRAIN_DATA_DIR.mkdir(parents=True, exist_ok=True)

sys.path.append(str(RETRIEVER_DIR))

import retriever_common as rc

SKIP_TOP_RANKS = 5

MAX_RANK = 50

NEGATIVES_PER_QUERY = 8

SAMPLING_STRATEGY = 'random'

RANDOM_SEED = 42

NEGATIVE_SOURCE_PATHS = {
    'bm25': BM25_OUTPUT_DIR / 'train_ranking.parquet',
}

rng = np.random.default_rng(RANDOM_SEED)

## **1. Read data**

In [ ]:
corpus = pd.read_parquet(DATA_DIR / 'corpus' / 'corpus.parquet')
query = pd.read_parquet(DATA_DIR / 'corpus' / 'queries.parquet')
qrels = pd.read_parquet(DATA_DIR / 'corpus' / 'qrels.parquet')

train_query = query[query['split'] == 'train'].reset_index(drop=True)
train_qrels = qrels[qrels['split'] == 'train']

positive_doc_ids_by_query = rc.build_qrels_map(train_qrels)

doc_id_to_text = dict(zip(corpus['doc_id'], corpus['embedding_text']))
query_id_to_text = dict(zip(train_query['query_id'], train_query['query']))

print(f"Corpus      : {len(corpus):,} passage")
print(f"Train query : {len(train_query):,}")
print(f"Train qrels : {len(train_qrels):,} cặp (query, doc)")

In [ ]:
positive_count_per_doc = train_qrels.groupby('doc_id').size()

probability_per_doc = (positive_count_per_doc / positive_count_per_doc.sum()).to_numpy()

collision_report = []

for batch_size in [32, 64, 128, 256]:
  expected_distinct = float((1.0 - (1.0 - probability_per_doc) ** batch_size).sum())

  collision_report.append({
      'batch_size': batch_size,
      'expected_distinct_positives': round(expected_distinct, 1),
      'expected_collisions': round(batch_size - expected_distinct, 1),
      'collision_ratio': round((batch_size - expected_distinct) / batch_size, 3),
  })

display(pd.DataFrame(collision_report))

print(f"\nSố passage từng xuất hiện làm positive: {len(positive_count_per_doc):,}")
print("=> Batch size KHÔNG được vượt quá con số này, và bắt buộc bật BatchSamplers.NO_DUPLICATES.")

## **2. Mine negative**

In [ ]:
def load_negative_candidates(source_paths: dict) -> pd.DataFrame:
  frames = []

  for source_name, path in source_paths.items():
    if not Path(path).exists():
      print(f"Bỏ qua '{source_name}': không tìm thấy {path}")
      continue

    ranking = pd.read_parquet(path)

    ranking = ranking[ranking['rank'].between(SKIP_TOP_RANKS + 1, MAX_RANK)].copy()

    ranking['source'] = source_name

    frames.append(ranking[['query_id', 'doc_id', 'rank', 'source']])

    print(f"'{source_name}': {len(ranking):,} ứng viên trong khoảng rank "
          f"[{SKIP_TOP_RANKS + 1}, {MAX_RANK}]")

  if not frames:
    raise FileNotFoundError("Không nạp được nguồn negative nào.")

  candidates = pd.concat(frames, ignore_index=True)

  candidates = (candidates.sort_values('rank')
                          .drop_duplicates(['query_id', 'doc_id'], keep='first')
                          .reset_index(drop=True))

  return candidates


negative_candidates = load_negative_candidates(NEGATIVE_SOURCE_PATHS)

print(f"\nTổng ứng viên sau khi gộp nguồn: {len(negative_candidates):,}")

In [ ]:
positive_pairs = set(zip(train_qrels['query_id'], train_qrels['doc_id']))

is_positive_mask = [
    (query_id, doc_id) in positive_pairs
    for query_id, doc_id in zip(negative_candidates['query_id'], negative_candidates['doc_id'])
]

removed_count = int(np.sum(is_positive_mask))

negative_candidates = negative_candidates[~np.array(is_positive_mask)].reset_index(drop=True)

print(f"Loại {removed_count:,} ứng viên vốn là positive của chính câu hỏi đó.")
print(f"Còn lại: {len(negative_candidates):,} ứng viên")

In [ ]:
candidate_doc_ids_by_query = (
    negative_candidates.sort_values('rank')
                       .groupby('query_id')['doc_id']
                       .apply(list)
                       .to_dict()
)

all_doc_ids = corpus['doc_id'].to_numpy()


def select_negatives(query_id: str) -> list[str]:
  candidate_doc_ids = candidate_doc_ids_by_query.get(query_id, [])

  if SAMPLING_STRATEGY == 'top':
    selected = candidate_doc_ids[:NEGATIVES_PER_QUERY]
  else:
    if len(candidate_doc_ids) <= NEGATIVES_PER_QUERY:
      selected = list(candidate_doc_ids)
    else:
      chosen_indices = rng.choice(len(candidate_doc_ids), size=NEGATIVES_PER_QUERY, replace=False)
      selected = [candidate_doc_ids[index] for index in sorted(chosen_indices)]

  if len(selected) < NEGATIVES_PER_QUERY:
    forbidden = positive_doc_ids_by_query.get(query_id, set()) | set(selected)

    available = [doc_id for doc_id in all_doc_ids if doc_id not in forbidden]

    shortage = min(NEGATIVES_PER_QUERY - len(selected), len(available))

    if shortage > 0:
      selected += [available[index] for index in rng.choice(len(available), size=shortage, replace=False)]

  return selected

In [ ]:
training_records = []

for query_id, positive_doc_ids in positive_doc_ids_by_query.items():
  negative_doc_ids = select_negatives(query_id)

  for positive_doc_id in sorted(positive_doc_ids):
    training_records.append({
        'query_id': query_id,
        'query': query_id_to_text[query_id],
        'positive_doc_id': positive_doc_id,
        'negative_doc_ids': negative_doc_ids,
        'number_of_negatives': len(negative_doc_ids),
    })

training_examples = pd.DataFrame(training_records)

print(f"Số mẫu huấn luyện: {len(training_examples):,}")
print(f"Số negative trung bình: {training_examples['number_of_negatives'].mean():.2f}")

display(training_examples['number_of_negatives'].value_counts().sort_index().to_frame('số mẫu'))

### **3.1. Check**

In [ ]:
for positive_doc_id, negative_doc_ids, query_id in zip(
    training_examples['positive_doc_id'], training_examples['negative_doc_ids'], training_examples['query_id']
):
  assert positive_doc_id not in negative_doc_ids, "Negative trùng positive."
  assert not (set(negative_doc_ids) & positive_doc_ids_by_query[query_id]), (
      "Negative trùng một positive khác của cùng câu hỏi."
  )

all_referenced_doc_ids = set(training_examples['positive_doc_id'])

for negative_doc_ids in training_examples['negative_doc_ids']:
  all_referenced_doc_ids |= set(negative_doc_ids)

assert all_referenced_doc_ids <= set(corpus['doc_id']), "Có doc_id không nằm trong corpus."

assert set(training_examples['query_id']) <= set(train_query['query_id'])

print("Các phép kiểm tra đã pass.")

In [ ]:
selected_pairs = set()

for query_id, negative_doc_ids in zip(training_examples['query_id'], training_examples['negative_doc_ids']):
  for doc_id in negative_doc_ids:
    selected_pairs.add((query_id, doc_id))

selected_ranks = negative_candidates[
    [(query_id, doc_id) in selected_pairs
     for query_id, doc_id in zip(negative_candidates['query_id'], negative_candidates['doc_id'])]
]['rank']

display(selected_ranks.describe(percentiles=[0.25, 0.5, 0.75]).to_frame('rank của negative được chọn'))

## **3. Export train data**

In [ ]:
training_examples.to_parquet(TRAIN_DATA_DIR / 'train_examples.parquet', index=False)

flat_records = []

for row in training_examples.itertuples(index=False):
  flat_records.append({
      'query_id': row.query_id, 'doc_id': row.positive_doc_id, 'label': 'positive', 'negative_index': -1
  })

  for negative_index, doc_id in enumerate(row.negative_doc_ids):
    flat_records.append({
        'query_id': row.query_id, 'doc_id': doc_id, 'label': 'negative', 'negative_index': negative_index
    })

train_pairs_flat = pd.DataFrame(flat_records)

train_pairs_flat.to_parquet(TRAIN_DATA_DIR / 'train_pairs_flat.parquet', index=False)

mining_config = {
    'skip_top_ranks': SKIP_TOP_RANKS,
    'max_rank': MAX_RANK,
    'negatives_per_query': NEGATIVES_PER_QUERY,
    'sampling_strategy': SAMPLING_STRATEGY,
    'random_seed': RANDOM_SEED,
    'negative_sources': [name for name in NEGATIVE_SOURCE_PATHS],
    'number_of_training_examples': len(training_examples),
    'max_safe_batch_size': int(len(positive_count_per_doc)),
}

with open(TRAIN_DATA_DIR / 'mining_config.json', 'w', encoding='utf-8') as file:
  json.dump(mining_config, file, ensure_ascii=False, indent=2)

print(json.dumps(mining_config, ensure_ascii=False, indent=2))